### 1. Background and Motivation: Why Was PEFT Researched?

In the early days of modern Transfer Learning (around 2018–2020), the standard paradigm for adapting a pre-trained language model (like BERT, RoBERTa, or GPT-2) to a specific downstream task (such as classification or sentiment analysis) was **Full Fine-Tuning (FFT)**. 

In FFT, all parameters of the pre-trained neural network are updated during backpropagation. This worked well when models were relatively small:
* **BERT-Base:** ~110 Million parameters (approx. 220 MB in 16-bit precision).
* **RoBERTa-Large:** ~355 Million parameters (approx. 710 MB in 16-bit precision).

Fine-tuning and saving multiple instances of these models was computationally manageable on standard consumer or enterprise GPUs.

#### The Scaling Paradigm Shift
As research showed that larger models yielded vastly superior capabilities, the industry transitioned to Large Language Models (LLMs) with billions or hundreds of billions of parameters (e.g., LLaMA, GPT-3, Mixtral). 

At this scale, Full Fine-Tuning became highly impractical for most organizations due to several critical bottlenecks:

1. **The Memory Wall (VRAM Bottleneck):** Training a model requires storing not just the weights, but also optimizer states, gradients, and forward activations. For a 70-billion parameter model in 16-bit precision, full fine-tuning requires over 1 Terabyte of GPU VRAM, placing it out of reach for standard hardware configurations.
2. **The Storage and Deployment Bottleneck:** If a company wants to serve 50 different downstream tasks (e.g., medical extraction, legal drafting, customer support, code generation), FFT forces them to store 50 separate copies of the fully fine-tuned 70B model. Storing and loading 50 different 140 GB models into GPU memory dynamically during production introduces unsustainable latency and storage costs.
3. **Catastrophic Forgetting:** Fine-tuning all weights on a narrow target dataset often degrades the general reasoning and world-knowledge capabilities that the model acquired during its massive pre-training phase.

To address these limitations, researchers sought methods to adapt models to specific tasks without modifying all of their parameters—leading to the development of **Parameter-Efficient Fine-Tuning (PEFT)**.

---

### 2. What Problem Is PEFT Solving, and How?

#### Conceptually
PEFT shifts the training paradigm from **global parameter updates** to **targeted parameter updates**. 

Let the pre-trained model parameters be represented by $\Theta_{0}$. 
* In **Full Fine-Tuning**, the final weights are $\Theta = \Theta_{0} + \Delta\Theta$. The optimization process updates every single element of the high-dimensional weight update tensor $\Delta\Theta$.
* In **PEFT**, the pre-trained weights $\Theta_{0}$ are frozen (meaning they require no gradient computation or optimizer tracking). PEFT introduces a significantly smaller set of task-specific parameters, denoted as $\theta$ (where $|\theta| \ll |\Theta_{0}|$, often less than 1% of the original parameter count). The optimization objective targets only these new parameters:

$$\theta^* = \arg\min_{\theta} \mathcal{L}(X; \Theta_{0}, \theta)$$

#### Mathematically (Memory Reduction)
To understand how this reduces memory footprint, let's look at what consumes GPU VRAM during training with the AdamW optimizer:
* **Model Weights (FP16/BF16):** 2 bytes per parameter.
* **Gradients (FP16/BF16):** 2 bytes per parameter.
* **AdamW Optimizer States (FP32):** 8 bytes per parameter (4 bytes for first momentum, 4 bytes for second momentum).

For a **70 Billion parameter model** under FFT, the memory required purely for training variables (excluding activations and temporary buffers) is:
$$\text{Memory} = 70\text{B} \times (2\text{ bytes} + 2\text{ bytes} + 8\text{ bytes}) = 840\text{ GB}$$

If we use a PEFT method like LoRA with a typical setting where we only train **0.5%** of the parameters ($350$ Million parameters), the calculation changes:
* **Frozen Base Model Weights (FP16):** $70\text{B} \times 2\text{ bytes} = 140\text{ GB}$ (gradients are not computed, optimizer states are not tracked for these weights).
* **Trainable PEFT Weights (FP16):** $350\text{M} \times 2\text{ bytes} = 0.7\text{ GB}$.
* **Trainable PEFT Gradients (FP16):** $350\text{M} \times 2\text{ bytes} = 0.7\text{ GB}$.
* **AdamW Optimizer States for PEFT (FP32):** $350\text{M} \times 8\text{ bytes} = 2.8\text{ GB}$.

$$\text{Total Memory} \approx 140\text{ GB} + 0.7\text{ GB} + 0.7\text{ GB} + 2.8\text{ GB} \approx 144.2\text{ GB}$$

By reducing the trainable parameters, the memory overhead drops from **840 GB** to **144.2 GB**, allowing the model to be fine-tuned on much smaller hardware configurations.

---

### 3. Classification of PEFT Methods

PEFT methods are generally grouped into three main families based on how they introduce and manipulate parameters:

```bash
                  ┌─────────────────────────────────────────┐
                  │              PEFT Families              │
                  └────────────────────┬────────────────────┘
                                       │
         ┌─────────────────────────────┼─────────────────────────────┐
         ▼                             ▼                             ▼
┌─────────────────┐           ┌─────────────────┐           ┌──────────────────┐
│Additive Methods │           │Selective Methods│           │Reparameterization│
└────────┬────────┘           └────────┬────────┘           └────────┬─────────┘
         │                             │                             │
         ├─ Adapters                   └─ BitFit                     ├─ LoRA
         ├─ Prompt Tuning                                            └─ QLoRA
         └─ Prefix Tuning
```

---

### 4. In-Depth Explanation of Key PEFT Methods

#### A. Reparameterization-Based: LoRA (Low-Rank Adaptation)
LoRA is one of the most widely adopted PEFT methods. It is built on the hypothesis that the weight updates during adaptation have a **"low intrinsic rank."** 

##### How it works:
Instead of updating the full pre-trained weight matrix $W_0 \in \mathbb{R}^{d \times k}$, LoRA decomposes the weight update $\Delta W$ into two low-rank matrices, $A$ and $B$, such that:

$$\Delta W = B \times A$$

Where $B \in \mathbb{R}^{d \times r}$ and $A \in \mathbb{R}^{r \times k}$. The rank $r$ is a hyperparameter chosen such that $r \ll \min(d, k)$.

```bash
                 Original Frozen Weight Matrix W0 (d x k)
                            ┌───────────────┐
                            │               │
                            │               │
                            │               │
                            └───────────────┘
                                    +
                 LoRA Adapter Matrices (BA)
                 Matrix B (d x r)       Matrix A (r x k)
                     ┌───┐
                     │   │   x   ┌───────────────────┐
                     │   │       │                   │
                     └───┘       └───────────────────┘
```

During training:
1. $W_0$ is completely frozen.
2. Matrix $A$ is typically initialized from a Gaussian distribution, and Matrix $B$ is initialized to 0. This ensures that at the start of training, $\Delta W = B \times A = 0$, meaning the model's behavior is initially unaltered.
3. For an input vector $x$, the forward pass calculation is:

$$h = W_0 x + \Delta W x = W_0 x + \frac{\alpha}{r} (BA) x$$

*(where $\alpha$ is a constant scaling hyperparameter).*

##### Numerical Example:
Consider a single linear projection layer inside a Transformer (e.g., the Query projection matrix $W_q$) with input/output dimensions $d = 4096$.
* **Full weight matrix $W_0$:** $4096 \times 4096 = 16,777,216$ parameters (approx. 16.8M).
* **Using LoRA with Rank $r = 8$:**
  * Matrix $A \in \mathbb{R}^{8 \times 4096} = 32,768$ parameters.
  * Matrix $B \in \mathbb{R}^{4096 \times 8} = 32,768$ parameters.
  * **Total trainable parameters for this layer:** $32,768 + 32,768 = 65,536$ parameters.

This reduces the trainable parameter count for this layer by **99.6%**.

##### Why LoRA has Zero Inference Latency:
When deploying the model to production, you do not need to keep the adapter matrices separate. Since addition is linear, you can mathematically merge the adapter weights back into the base weights:

$$W_{\text{deployed}} = W_0 + \frac{\alpha}{r} (BA)$$

You compute $W_{\text{deployed}}$ once prior to serving, replace $W_0$ in memory, and run inference at the exact same latency as the unmodified base model.

---

#### B. Reparameterization-Based: QLoRA (Quantized Low-Rank Adaptation)
QLoRA is an evolution of LoRA designed to compress the baseline memory footprint of the frozen weights during training.

##### How it works:
QLoRA introduces three distinct features to maximize hardware efficiency:
1. **4-bit NormalFloat (NF4) Quantization:** Standard quantization methods scale values linearly. NF4 is an information-theoretically optimal quantization format designed specifically for normally distributed data (which neural network weights usually are). It compresses the frozen base model weights $W_0$ from 16-bit precision to 4-bit precision with minimal loss in model capability.
2. **Double Quantization (DQ):** Quantization constants (the scaling factors used to convert 16-bit to 4-bit) also take up memory. DQ quantizes those constants as well, saving roughly 0.37 bits per parameter.
3. **Paged Optimizers:** If a memory spike occurs (e.g., during a forward pass with a long sequence), CUDA Unified Memory is used to page optimizer state memory out to physical system RAM and back. This helps prevent sudden Out-of-Memory (OOM) failures.

During the training backward pass, the 4-bit weights are dequantized to 16-bit computation precision on-the-fly, gradients are calculated for the high-precision LoRA adapters ($A$ and $B$), and the dequantized weights are immediately discarded from cache.

---

#### C. Additive Methods: Adapters (e.g., Houlsby Adapters)
Adapters were one of the earliest PEFT approaches. Instead of altering existing projection matrices, they introduce small, localized neural network modules directly inside the Transformer layer architecture.

##### How it works:
An adapter is a bottleneck network placed after the Multi-Head Attention and Feed-Forward layers. 
The high-dimensional representation $h \in \mathbb{R}^d$ is projected down to a low-dimensional bottleneck space $r$, a non-linear activation function is applied, and it is then projected back up to $d$.

$$\text{Adapter}(h) = h + f(h W_{\text{down}}) W_{\text{up}}$$

Where:
* $W_{\text{down}} \in \mathbb{R}^{d \times r}$
* $W_{\text{up}} \in \mathbb{R}^{r \times d}$
* $f(\cdot)$ is a non-linear activation function (such as GeLU).

```bash
         Standard Transformer Block                 With Adapter Insertion
           ┌───────────────────────┐               ┌───────────────────────┐
           │ Multi-Head Attention  │               │ Multi-Head Attention  │
           └───────────┬───────────┘               └───────────┬───────────┘
                       │                                       │
                       ▼                                       ▼
           ┌───────────────────────┐               ┌───────────────────────┐
           │      LayerNorm        │               │  Adapter Layer (New)  │
           └───────────┬───────────┘               └───────────┬───────────┘
                       │                                       │
                       ▼                                       ▼
           ┌───────────────────────┐               ┌───────────────────────┐
           │ Feed-Forward Network  │               │      LayerNorm        │
           └───────────────────────┘               └───────────────────────┘
```

##### Drawback:
Unlike LoRA, Adapters cannot be easily merged back into the base model weights because they are sequenced between non-linear activation steps. This introduces additional sequential steps in the forward pass, which increases **inference latency**.

---

#### D. Additive Methods: Prompt Tuning and Prefix Tuning

##### Prompt Tuning (Lester et al.)
Instead of modifying model weights, Prompt Tuning prepends a small set of trainable, continuous vector embeddings (often called "soft prompts" or "virtual tokens") to the input sequence.

```bash
Input Tokens:     [ "Translate", "this", "sentence" ]
Soft Prompts:     [ P1, P2, P3, P4 ]  <-- Only these vectors are updated during training
Actual Input:     [ P1, P2, P3, P4, "Translate", "this", "sentence" ]
```

During training, the base model's embedding matrix and layer weights are kept frozen. Only the continuous embeddings $P_i$ are optimized via gradient descent.

##### Prefix Tuning (Li & Liang)
Prefix Tuning extends this concept. Instead of appending virtual tokens only at the input layer, it prepends learnable continuous vectors directly to the keys ($K$) and values ($V$) at **every** self-attention layer of the Transformer.

$$K_{\text{new}} = [P_K; K], \quad V_{\text{new}} = [P_V; V]$$

This allows subsequent layers to attend to these virtual prefixes as if they were real context tokens, guiding the model's generation process without altering its core weights.

##### Drawback:
These virtual tokens consume space in the model’s limited context window (maximum sequence length), leaving less room for the actual prompt and response.

---

#### E. Selective Methods: BitFit
BitFit is a minimalist, selective PEFT method. 

##### How it works:00
Instead of adding new parameters or matrices, BitFit freezes the main weight matrices ($W$) and only updates the **bias terms ($b$)** across the entire network. 

$$y = W x + b \quad (\text{Only } b \text{ is updated})$$

Bias terms usually account for less than **0.1%** of a model's total parameter count. While highly parameter-efficient and useful for simpler classification tasks, BitFit's performance can degrade when scaling to complex generation tasks on larger models compared to representation-rich methods like LoRA.

---

### 5. Comparative Summary of PEFT Methods

| Method | Type | Trainable Params (%) | Inference Latency | Core Advantage | Primary Limitation |
| :--- | :--- | :--- | :--- | :--- | :--- |
| **Full Fine-Tuning** | Baseline | 100% | None | Maximum learning capacity on highly specialized domains. | Extremely high VRAM and storage requirements. |
| **Adapters** | Additive | 1% – 5% | Increased | Highly modular and modularly interchangeable. | Sequential architecture increases inference latency. |
| **Prompt Tuning** | Additive | 0.01% – 0.1% | Minimal | Minimal training parameter requirements. | Consumes portion of input context length. |
| **Prefix Tuning** | Additive | 0.1% – 1% | Minimal | Good performance on generation tasks. | Consumes portion of input context length. |
| **LoRA** | Reparameterization | 0.1% – 1% | None (if merged) | No inference latency; performance is generally on par with full tuning. | Slightly more complex setup compared to prompt tuning. |
| **QLoRA** | Reparameterization | 0.1% – 1% | None (if merged) | Drastically reduces VRAM requirements (can train 70B models on smaller hardware). | Marginally slower training speed due to on-the-fly dequantization. |

<hr>
<hr>


## **Adapters**

### Part 1: VRAM Breakdown for BERT-Base Full Fine-Tuning (FFT)

To calculate the actual GPU VRAM required to fully fine-tune BERT-Base (110M parameters) using a standard optimizer like **AdamW**, we categorize VRAM usage into two types: **Static Memory** (determined solely by the model's parameters) and **Dynamic Memory** (determined by the batch size and sequence length).

#### 1. Static VRAM Overhead (Using Mixed Precision FP16/FP32)
In modern deep learning, models are trained using mixed-precision. The base model weights and gradients are kept in 16-bit (FP16 or BF16) to save memory, while the optimizer maintains high-precision 32-bit (FP32) copies of the weights and states to prevent numerical underflow.

For **BERT-Base (110M parameters)**:
* **Model Weights (FP16):** $110\text{M} \times 2\text{ bytes} = \mathbf{220\text{ MB}}$
* **Gradients (FP16):** $110\text{M} \times 2\text{ bytes} = \mathbf{220\text{ MB}}$
* **AdamW Optimizer States (FP32):** AdamW tracks three 32-bit tensors per parameter: the master FP32 weights, the first momentum vector, and the second momentum vector.
  $$\text{Optimizer State} = 110\text{M} \times (4\text{ bytes (weights)} + 4\text{ bytes (momentum)} + 4\text{ bytes (variance)}) = \mathbf{1.32\text{ GB}}$$

$$\text{Total Static VRAM for FFT} = 220\text{ MB} + 220\text{ MB} + 1.32\text{ GB} = \mathbf{1.76\text{ GB}}$$

#### 2. Dynamic VRAM Overhead (Activations)
To compute gradients during the backward pass, the GPU must store the intermediate outputs of every single layer (activations) during the forward pass. 
* For a standard batch size of $32$ and a sequence length of $512$, BERT-Base requires roughly **$4\text{ GB}$ to $6\text{ GB}$** of activation memory, depending on the implementation details (such as whether gradient checkpointing is used).

#### Actual GPU Requirement for BERT-Base FFT:
* **Minimum VRAM:** Around **$6\text{ GB}$ to $8\text{ GB}$** of VRAM.
* **Suitable GPUs:** A single consumer GPU like an NVIDIA RTX 3060/4060 (12GB) or an enterprise NVIDIA T4 (16GB) is sufficient. 

*Note: While BERT-Base is small enough to train on basic hardware, this scale calculation becomes critical as models grow. For a 7 Billion parameter model (e.g., LLaMA-7B), the static VRAM for FFT is $7\text{B} \times 16\text{ bytes} = \mathbf{112\text{ GB}}$, requiring multiple high-end enterprise GPUs just to hold the model and its optimizer states.*

---

### Part 2: The Adapter Scenario — Where Do the Savings Come From?

Your primary question is: **If we still need to store the frozen base model weights in VRAM for the forward pass, how does this save VRAM?**

The efficiency gain does not come from removing the base model weights. You are correct that we must keep the frozen base model weights in VRAM for the forward pass ($220\text{ MB}$). **The savings come entirely from eliminating the gradients and optimizer states for those frozen layers.**

Let's look at the math for BERT-Base with Houlsby Adapters ($2.38\text{M}$ trainable parameters):

| Category | Full Fine-Tuning (FFT) | Adapter Fine-Tuning | Memory Savings |
| :--- | :--- | :--- | :--- |
| **Base Model Weights (FP16)** | $220\text{ MB}$ (Trainable) | $215.24\text{ MB}$ (Frozen) | None (Still in VRAM) |
| **Adapter Weights (FP16)** | $0\text{ MB}$ | $4.76\text{ MB}$ (Trainable) | $-4.76\text{ MB}$ |
| **Gradients (FP16)** | $220\text{ MB}$ (For all weights) | **$4.76\text{ MB}$** (Only for Adapters) | **$+215.24\text{ MB}$** |
| **Optimizer States (FP32)** | $1.32\text{ GB}$ (For all weights) | **$28.56\text{ MB}$** (Only for Adapters) | **$+1.29\text{ GB}$** |
| **Total Static VRAM** | **$1.76\text{ GB}$** | **$253.32\text{ MB}$** | **$+1.50\text{ GB}$** |

#### Why this scales:
For BERT-Base, we save **$1.5\text{ GB}$** of static VRAM. 
For a **7 Billion parameter model**:
* **FFT static VRAM:** $112\text{ GB}$ (Requires multiple $80\text{GB}$ A100 GPUs)
* **Adapter static VRAM:** Base Model weights ($14\text{ GB}$) + Adapter Overhead ($0.56\text{ GB}$) = **$14.56\text{ GB}$** (Fits easily on a single consumer RTX 3090 or RTX 4090 GPU).

---

### Part 3: The Gradient Propagation Question

Your next core question is: **To update the adapter weights, don't we need to store the gradients of the blocks in front of the adapter network?**

To understand why this is highly efficient, we must distinguish between two fundamentally different types of gradients during backpropagation:
1. **Weight Gradients ($\frac{\partial \mathcal{L}}{\partial W}$):** The gradients with respect to the actual trainable parameters of a layer. These must be stored in memory to update the weights.
2. **Activation Gradients ($\frac{\partial \mathcal{L}}{\partial h}$):** The gradients with respect to the intermediate layer outputs. These are used only to pass information backward through the network.

#### How Backpropagation Works in a Frozen Layer
Consider a frozen base model layer ($W_{\text{base}}$) located immediately before a trainable adapter layer ($W_{\text{adapter}}$):

```bash
[Previous Layers] ---> [Frozen Layer: W_base] ---> [Trainable Adapter: W_adapter] ---> [Loss: L]
```

During the backward pass:
1. We compute the activation gradient at the adapter layer: $\frac{\partial \mathcal{L}}{\partial \text{Adapter}(h)}$.
2. We compute the weight gradient for the adapter: $\frac{\partial \mathcal{L}}{\partial W_{\text{adapter}}}$. This is **saved in VRAM** to update the adapter weights.
3. To continue backpropagation to preceding layers, we compute the activation gradient of the frozen layer:
   $$\frac{\partial \mathcal{L}}{\partial h_{\text{frozen}}} = \frac{\partial \mathcal{L}}{\partial \text{Adapter}(h)} \cdot W_{\text{base}}^T$$
4. **Crucial Step:** We completely **skip** the step that computes the gradient with respect to the frozen layer's weights:
   $$\frac{\partial \mathcal{L}}{\partial W_{\text{base}}} = \text{Not Computed / Not Stored}$$

#### How this saves memory:
* **Weight gradients ($\frac{\partial \mathcal{L}}{\partial W_{\text{base}}}$)** are large tensors that match the exact shape of the base model weights. By freezing the base model, we do not allocate any VRAM for these tensors.
* **Activation gradients ($\frac{\partial \mathcal{L}}{\partial h}$)** are transient vectors of size `[Batch, SeqLen, HiddenDim]`. The GPU computes them, uses them immediately to pass the gradient to the previous layer, and then **frees them from memory on-the-fly**. They are not stored persistently in the gradient buffer.

Because we do not allocate persistent memory for the base model's weight gradients or optimizer states, we achieve significant VRAM savings.

---

### Part 4: Disk Storage and Multi-Task Deployment

Apart from VRAM during training, PEFT solves a massive bottleneck in **disk storage** and **GPU deployment** for production pipelines.

#### Scenario: An enterprise serving 100 different tasks (e.g., customized models for 100 different clients)

#### 1. Under Full Fine-Tuning (FFT)
If you train 100 fully fine-tuned BERT models:
* Each model is saved as a complete set of weights: $110\text{M} \times 4\text{ bytes (FP32)} = 440\text{ MB}$.
* Total disk storage required: $100 \times 440\text{ MB} = \mathbf{44\text{ GB}}$.
* **The Deployment Problem:** To serve these 100 tasks on a GPU, you must load all 44 GB of models into VRAM, or dynamically swap 440 MB files in and out of the GPU whenever a client sends a request. This causes high latency and requires expensive hardware.

#### 2. Under Adapter Fine-Tuning
If you train 100 adapters:
* You save only **one** copy of the frozen base model: $440\text{ MB}$.
* For each task, you save only the adapter weights ($2.38\text{M}$ parameters): $2.38\text{M} \times 4\text{ bytes} \approx 9.5\text{ MB}$ per adapter.
* Total disk storage required: $440\text{ MB} + (100 \times 9.5\text{ MB}) = \mathbf{1.39\text{ GB}}$.

```bash
Disk Storage Footprint:
FFT (100 Tasks):      [=============================================] 44 GB
Adapters (100 Tasks): [=] 1.39 GB
```

#### How it works in GPU Production:
Instead of running 100 separate model instances, you load **one single copy** of the base model BERT-Base ($440\text{ MB}$) into the GPU's memory. 

You can then keep all 100 adapters ($100 \times 9.5\text{ MB} \approx 950\text{ MB}$) in the same GPU VRAM. When a request for Task A comes in, the inputs are routed through the shared base model and the tiny Adapter A weights. When a request for Task B comes in, the inputs are routed through the same base model and Adapter B weights. 

This enables you to serve 100 customized tasks on a single GPU with almost no switching overhead.

<hr>
<hr>